In [1]:
# 00_data_generation.ipynb
# Simulation data collection for the metamorphic board-oversight study.
# Generates synthetic board-level respondents who judge boundary-case vignettes
# arranged in metamorphic pairs, under a two-stage design (individual judgment ->
# coordination intervention -> revised judgment), with a known ground-truth class
# per board. All randomness is seeded for reproducibility. No file paths are printed.
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

SEED = 20261026
rng = np.random.default_rng(SEED)
DATA = Path("..") / "data"
DATA.mkdir(parents=True, exist_ok=True)

# ----- Structural parameters -----
N_BOARDS, N_SCENARIOS = 140, 12
RELATIONS = ["monotonicity", "equivalence", "ordering", "composition"]
R_MIN, R_MAX = 1, 7
def zdraw(n): return rng.normal(0, 1, n)

# ----- Board characteristics -----
ai_literacy  = zdraw(N_BOARDS)
diversity    = np.clip(rng.beta(2, 2, N_BOARDS), 0.02, 0.98)
independence = np.clip(rng.beta(2.5, 2, N_BOARDS), 0.02, 0.98)
ceo_tenure   = np.clip(rng.gamma(2.0, 3.0, N_BOARDS), 0.5, 25.0)
risk_culture = zdraw(N_BOARDS)
escalation   = rng.binomial(1, 0.5, N_BOARDS)
board_size   = rng.integers(5, 16, N_BOARDS)
tenure_z = (ceo_tenure - ceo_tenure.mean()) / ceo_tenure.std()

# ----- Latent properties the metrics must recover -----
D_lin = 2*0.9*(diversity-0.5) - 0.5*ai_literacy + 0.35*zdraw(N_BOARDS)
D_b = 1/(1+np.exp(-D_lin))                                  # Stage-1 dispersion tendency
C_lin = (1.1*escalation + 2*1.0*(independence-0.5) + 2*0.7*(diversity-0.5)
         + 0.6*tenure_z + 0.3*risk_culture + 0.35*zdraw(N_BOARDS))
C_b = 1/(1+np.exp(-C_lin))                                  # convergence capacity
hi_disp, hi_conv = D_b >= np.median(D_b), C_b >= np.median(C_b)
gt_class = np.where(~hi_disp, "A_consensus",
            np.where(hi_conv, "B_pluralism", "C_failure"))
gt_failure = (gt_class == "C_failure").astype(int)

# ----- Respondents nested in boards (uneven; some boards have n=1) -----
resp_per_board = rng.integers(1, 8, N_BOARDS)
board_of = np.repeat(np.arange(N_BOARDS), resp_per_board)
N = board_of.size
resp_ai   = ai_literacy[board_of] + 0.5*zdraw(N)
resp_bias = 0.36*(0.5*risk_culture[board_of] + 0.8*zdraw(N))

# ----- Scenario severities and metamorphic partners -----
scen_sev = rng.uniform(2.0, 6.0, N_SCENARIOS)
scen_rel = np.array([RELATIONS[i % len(RELATIONS)] for i in range(N_SCENARIOS)])
dl = rng.uniform(0.8, 1.6, N_SCENARIOS)
follow_sev = scen_sev.copy()
for i, r in enumerate(scen_rel):
    if r == "monotonicity":  follow_sev[i] = scen_sev[i] - dl[i]
    elif r == "ordering":    follow_sev[i] = scen_sev[i] + dl[i]
    elif r == "composition": follow_sev[i] = scen_sev[i] + 0.5*dl[i]
follow_sev = np.clip(follow_sev, 1.0, 7.0)

def noisy(true_val):
    sd = np.clip(0.35 + 1.9*D_b[board_of] - 0.35*resp_ai, 0.2, 3.0)
    return np.clip(np.rint(true_val + resp_bias + rng.normal(0, sd)), R_MIN, R_MAX)
def board_anchor(vals):
    return pd.Series(vals).groupby(board_of).transform("mean").to_numpy()

frames, w = [], C_b[board_of]
for s in range(N_SCENARIOS):
    src1, fol1 = noisy(scen_sev[s]), noisy(follow_sev[s])
    src2 = np.clip(np.rint((1-w)*src1 + w*board_anchor(src1) + rng.normal(0,0.25,N)), R_MIN, R_MAX)
    fol2 = np.clip(np.rint((1-w)*fol1 + w*board_anchor(fol1) + rng.normal(0,0.25,N)), R_MIN, R_MAX)
    for st,(rs,rf) in [(1,(src1,fol1)),(2,(src2,fol2))]:
        frames.append(pd.DataFrame({"board_id": board_of, "resp_id": np.arange(N),
            "pair_id": s, "relation": scen_rel[s], "stage": st,
            "rating_source": rs.astype(int), "rating_follow": rf.astype(int)}))
long = pd.concat(frames, ignore_index=True)

# ----- Metamorphic violation (vectorized) -----
rel = long["relation"].to_numpy()
src = long["rating_source"].to_numpy(); fol = long["rating_follow"].to_numpy()
v = np.zeros(len(long), int)
v[(rel=="monotonicity") & (fol>src)] = 1   # follow-up must not exceed source
v[(rel=="equivalence")  & (fol!=src)] = 1   # follow-up must equal source
v[(rel=="ordering")     & (fol<src)] = 1    # follow-up must be at least source
v[(rel=="composition")  & (fol<src)] = 1
long["violation"] = v

# ----- Board-level analysis table -----
def disp(df): return df.groupby("pair_id")["rating_source"].std(ddof=0).mean()
recs = []
for b, g in long.groupby("board_id"):
    s1, s2 = g[g.stage==1], g[g.stage==2]
    v1, v2 = s1["violation"].mean(), s2["violation"].mean()
    d1, d2 = disp(s1), disp(s2)
    d1 = 0.0 if np.isnan(d1) else d1; d2 = 0.0 if np.isnan(d2) else d2
    recs.append(dict(board_id=b, viol1=v1, viol2=v2, disp1=d1, disp2=d2,
        conv_disp=(d1-d2)/d1 if d1>0 else 0.0, conv_viol=(v1-v2)/v1 if v1>0 else 0.0,
        n_resp=int((g.stage==1).sum() // N_SCENARIOS)))
board = pd.DataFrame(dict(board_id=np.arange(N_BOARDS), ai_literacy=ai_literacy,
    diversity=diversity, independence=independence, ceo_tenure=ceo_tenure,
    ceo_tenure_z=tenure_z, risk_culture=risk_culture, escalation=escalation,
    board_size=board_size, D_b=D_b, C_b=C_b, gt_class=gt_class, gt_failure=gt_failure)
    ).merge(pd.DataFrame(recs), on="board_id")

long.to_csv(DATA/"respondent_long.csv", index=False)
board.to_csv(DATA/"board_analysis.csv", index=False)

# ----- Reproducibility summary (no paths) -----
print(f"seed={SEED}  boards={N_BOARDS}  respondents={N}  scenarios={N_SCENARIOS}")
print("respondents per board: min=%d median=%d max=%d" %
      (resp_per_board.min(), int(np.median(resp_per_board)), resp_per_board.max()))
print(board["gt_class"].value_counts().to_string())
print("files written: respondent_long.csv, board_analysis.csv")


seed=20261026  boards=140  respondents=582  scenarios=12
respondents per board: min=1 median=4 max=7
gt_class
A_consensus    70
B_pluralism    38
C_failure      32
files written: respondent_long.csv, board_analysis.csv
